# AI-Powered Customer Call Analytics

## Objective
Convert customer call recordings into structured business insights using:
- Speech-to-Text (Whisper)
- Sentiment Analysis
- NLP
- Topic Classification
- Power BI Dashboard

In [20]:
import os
import pandas as pd
import spacy
from textblob import TextBlob

In [21]:
# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Load transcript dataset
df = pd.read_csv("../structured_data/call_data.csv")

df.head()

,Call_ID,Transcript,Word_Count
0,call_recording_01,"Hello, I'm Sarah Miller. I'm calling to inqui...",45
1,call_recording_02,I am extremely dissatisfied with my recent or...,73
2,call_recording_03,"Hi, this is Maria Rodriguez. I'm having troub...",60
3,call_recording_04,I just wanted to call and say how pleased I a...,62
4,call_recording_05,"Hello, my name is Jessica Brown. I'd like to ...",52


In [22]:
def get_sentiment(text):

    score = round(TextBlob(str(text)).sentiment.polarity, 2)

    if score > 0:
        sentiment = "Positive"
    elif score < 0:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    return pd.Series([score, sentiment])


df[["Sentiment_Score", "Sentiment"]] = df["Transcript"].apply(get_sentiment)

df.head()

,Call_ID,Transcript,Word_Count,Sentiment_Score,Sentiment
0,call_recording_01,"Hello, I'm Sarah Miller. I'm calling to inqui...",45,-0.01,Negative
1,call_recording_02,I am extremely dissatisfied with my recent or...,73,0.05,Positive
2,call_recording_03,"Hi, this is Maria Rodriguez. I'm having troub...",60,-0.05,Negative
3,call_recording_04,I just wanted to call and say how pleased I a...,62,0.48,Positive
4,call_recording_05,"Hello, my name is Jessica Brown. I'd like to ...",52,0.00,Neutral


In [23]:
def summarize(text):

    sentences = str(text).split(".")

    return ".".join(sentences[:2])


df["Summary"] = df["Transcript"].apply(summarize)

df.head()

,Call_ID,Transcript,Word_Count,Sentiment_Score,Sentiment,Summary
0,call_recording_01,"Hello, I'm Sarah Miller. I'm calling to inqui...",45,-0.01,Negative,"Hello, I'm Sarah Miller. I'm calling to inqui..."
1,call_recording_02,I am extremely dissatisfied with my recent or...,73,0.05,Positive,I am extremely dissatisfied with my recent or...
2,call_recording_03,"Hi, this is Maria Rodriguez. I'm having troub...",60,-0.05,Negative,"Hi, this is Maria Rodriguez. I'm having troub..."
3,call_recording_04,I just wanted to call and say how pleased I a...,62,0.48,Positive,I just wanted to call and say how pleased I a...
4,call_recording_05,"Hello, my name is Jessica Brown. I'd like to ...",52,0.00,Neutral,"Hello, my name is Jessica Brown. I'd like to ..."


In [24]:
CUSTOM_STOPWORDS = {
    "hello","hi","thanks","thank","please",
    "okay","ok","yeah","yes","sir","madam",
    "customer","agent","call","calling",
    "today","information","help","service",
    "support","company","phone","number"
}

def extract_keywords(text):

    doc = nlp(str(text))

    keywords = []

    # Noun phrases
    for chunk in doc.noun_chunks:

        phrase = chunk.text.lower().strip()

        if (
            len(phrase) > 3
            and phrase not in CUSTOM_STOPWORDS
            and phrase not in keywords
        ):
            keywords.append(phrase)

    # Individual nouns
    for token in doc:

        word = token.lemma_.lower()

        if (
            token.pos_ in ["NOUN", "PROPN"]
            and len(word) > 2
            and word not in CUSTOM_STOPWORDS
            and word not in keywords
        ):
            keywords.append(word)

    return ", ".join(keywords[:20])


df["Keywords"] = df["Transcript"].apply(extract_keywords)

df.head()

,Call_ID,Transcript,Word_Count,Sentiment_Score,Sentiment,Summary,Keywords
0,call_recording_01,"Hello, I'm Sarah Miller. I'm calling to inqui...",45,-0.01,Negative,"Hello, I'm Sarah Miller. I'm calling to inqui...","sarah miller, the ac-7892 air conditioner unit..."
1,call_recording_02,I am extremely dissatisfied with my recent or...,73,0.05,Positive,I am extremely dissatisfied with my recent or...,"my recent order, john davis, order number, the..."
2,call_recording_03,"Hi, this is Maria Rodriguez. I'm having troub...",60,-0.05,Negative,"Hi, this is Maria Rodriguez. I'm having troub...","this, maria rodriguez, trouble, my lap2110 lap..."
3,call_recording_04,I just wanted to call and say how pleased I a...,62,0.48,Positive,I just wanted to call and say how pleased I a...,"the dw6543 dishwasher, my name, robert smith, ..."
4,call_recording_05,"Hello, my name is Jessica Brown. I'd like to ...",52,0.00,Neutral,"Hello, my name is Jessica Brown. I'd like to ...","my name, jessica brown, an order, one ov1357 o..."


In [25]:
TOPIC_KEYWORDS = {

    "Product Inquiry": [
        "product","device","speaker","camera",
        "printer","tablet","laptop","television",
        "tv","monitor","medical","router",
        "air conditioner","microwave",
        "dishwasher","oven","console",
        "game","feature","specification"
    ],

    "Order & Purchase": [
        "order","buy","purchase","price",
        "delivery","shipping","tracking",
        "stock","payment","discount",
        "quote","availability"
    ],

    "Customer Support": [
        "problem","issue","complaint",
        "refund","replacement","exchange",
        "delay","damaged","broken",
        "support","error","assistance"
    ]

}

def classify_topic(text):

    text = str(text).lower()

    scores = {}

    for topic, words in TOPIC_KEYWORDS.items():

        scores[topic] = sum(word in text for word in words)

    return max(scores, key=scores.get)


df["Topic"] = df["Keywords"].apply(classify_topic)

df["Topic"].value_counts()

Topic
Product Inquiry     11
Order & Purchase     8
Customer Support     1
Name: count, dtype: int64

In [26]:
df["Conversation_Size"] = pd.qcut(
    df["Word_Count"],
    q=3,
    labels=["Small", "Medium", "Large"]
)

df["Conversation_Size"].value_counts()

Conversation_Size
Small     9
Large     7
Medium    4
Name: count, dtype: int64

In [27]:
df.head()

,Call_ID,Transcript,Word_Count,Sentiment_Score,Sentiment,Summary,Keywords,Topic,Conversation_Size
0,call_recording_01,"Hello, I'm Sarah Miller. I'm calling to inqui...",45,-0.01,Negative,"Hello, I'm Sarah Miller. I'm calling to inqui...","sarah miller, the ac-7892 air conditioner unit...",Product Inquiry,Small
1,call_recording_02,I am extremely dissatisfied with my recent or...,73,0.05,Positive,I am extremely dissatisfied with my recent or...,"my recent order, john davis, order number, the...",Order & Purchase,Large
2,call_recording_03,"Hi, this is Maria Rodriguez. I'm having troub...",60,-0.05,Negative,"Hi, this is Maria Rodriguez. I'm having troub...","this, maria rodriguez, trouble, my lap2110 lap...",Product Inquiry,Medium
3,call_recording_04,I just wanted to call and say how pleased I a...,62,0.48,Positive,I just wanted to call and say how pleased I a...,"the dw6543 dishwasher, my name, robert smith, ...",Product Inquiry,Medium
4,call_recording_05,"Hello, my name is Jessica Brown. I'd like to ...",52,0.00,Neutral,"Hello, my name is Jessica Brown. I'd like to ...","my name, jessica brown, an order, one ov1357 o...",Order & Purchase,Small


In [28]:
df.to_csv(
    "../structured_data/final_call_analytics.csv",
    index=False
)

print("✅ Final CSV Saved Successfully")

✅ Final CSV Saved Successfully


In [29]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTopic Distribution:")
print(df["Topic"].value_counts())

print("\nSentiment Distribution:")
print(df["Sentiment"].value_counts())

print("\nConversation Size:")
print(df["Conversation_Size"].value_counts())

Dataset Shape: (20, 9)

Columns:
Index(['Call_ID', 'Transcript', 'Word_Count', 'Sentiment_Score', 'Sentiment',
       'Summary', 'Keywords', 'Topic', 'Conversation_Size'],
      dtype='object')

Missing Values:
Call_ID              0
Transcript           0
Word_Count           0
Sentiment_Score      0
Sentiment            0
Summary              0
Keywords             0
Topic                0
Conversation_Size    0
dtype: int64

Topic Distribution:
Topic
Product Inquiry     11
Order & Purchase     8
Customer Support     1
Name: count, dtype: int64

Sentiment Distribution:
Sentiment
Negative    10
Positive     9
Neutral      1
Name: count, dtype: int64

Conversation Size:
Conversation_Size
Small     9
Large     7
Medium    4
Name: count, dtype: int64


In [ ]:
from sqlalchemy import create_engine

username = "postgres"
password = "sanju123"
host = "localhost"
port = "1234"
database = "customer_call_analytics"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)
df.columns = df.columns.str.lower()
df.to_sql(
    "customer_calls",
    engine,
    if_exists="replace",
    index=False
)

print("✅ Uploaded to PostgreSQL Successfully!")

✅ Uploaded to PostgreSQL Successfully!


: 